### General Results of the optimisation

In [ ]:
import pandas as pd
import json
from pathlib import Path
import matplotlib.pyplot as plt

# Get latest run
RESULTS_ROOT = Path("02-MODEL-RESULTS")
latest_run = sorted([d for d in RESULTS_ROOT.iterdir() if d.is_dir()])[-1]

# Load summary
with open(latest_run / "results_summary.json", "r") as f:
    summary = json.load(f)

# Load time series
df = pd.read_csv(latest_run / "timeseries_results.csv", parse_dates=['timestamp'])
df_fin = pd.read_csv(latest_run / "financial_cashflows.csv")

print(f"Loaded results from: {latest_run.name}")

In [ ]:
from IPython.display import display, Markdown

kpi_metrics = ["npv", "payback_years", "objective_total_cost", "battery_capacity_kwh", "import_cost"]
kpi_df = pd.DataFrame([{"Metric": k, "Value": summary.get(k)} for k in kpi_metrics])

display(Markdown("## Optimization KPIs"))
display(kpi_df.style.format({"Value": "{:,.2f}"}))

In [ ]:
## 2. Battery State of Charge (Seasonal Samples)
fig, axes = plt.subplots(2, 2, figsize=(15, 10), sharey=True)
axes = axes.flatten()
dates = [f"{df_soc['timestamp'].dt.year.iloc[0]}-01-01", 
         f"{df_soc['timestamp'].dt.year.iloc[0]}-04-01", 
         f"{df_soc['timestamp'].dt.year.iloc[0]}-07-01", 
         f"{df_soc['timestamp'].dt.year.iloc[0]}-10-01"]

for i, start_date in enumerate(dates):
    start = pd.to_datetime(start_date)
    end = start + pd.Timedelta(days=7)
    mask = (df_soc['timestamp'] >= start) & (df_soc['timestamp'] < end)
    
    ax = axes[i]
    if mask.any():
        ax.plot(df_soc.loc[mask, 'timestamp'], df_soc.loc[mask, 'battery_soc'], color='#2ca02c', lw=2)
        ax.fill_between(df_soc.loc[mask, 'timestamp'], df_soc.loc[mask, 'battery_soc'], alpha=0.2, color='#2ca02c')
        ax.set_title(f"Week of {start.strftime('%B %d')}")
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, "No Data", ha='center')

plt.suptitle("Battery State of Charge (SOC) - Seasonal Samples", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
## 3. Financial Projection
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.bar(df_fin['year'], df_fin['cashflow'], color='skyblue', label='Annual Cashflow (CHF)')
ax1.set_xlabel('Year')
ax1.set_ylabel('Annual Cashflow [CHF]')

ax2 = ax1.twinx()
ax2.plot(df_fin['year'], df_fin['discounted_cashflow'].cumsum(), color='red', marker='o', label='Cumulative NPV')
ax2.axhline(0, color='black', linewidth=1, linestyle='--')
ax2.set_ylabel('Cumulative NPV [CHF]')

plt.title("Investment Lifecycle & Payback Analysis")
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
ax1.grid(axis='y', alpha=0.3)
plt.show()

display(Markdown(f"**Payback Period:** {summary.get('payback_years', 'N/A')} years"))
display(Markdown(f"**Net Present Value (NPV):** CHF {summary.get('npv', 0):,.2f}"))